# Metrics Example

This notebook demonstrates how to use the evaluation metrics module. The metrics are divided into three groups:

1. **Ranking Metrics** - Evaluate how well the model ranks relevant items (nDCG, MAP, Precision@k, Recall@k)
2. **Rating Metrics** - Evaluate rating prediction accuracy (RMSE, MAE, R², Explained Variance)
3. **Beyond-Accuracy Metrics** - Evaluate recommendation quality beyond accuracy (Coverage, Novelty)

In [4]:
import sys
import warnings
import pandas

sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

from src.evaluation.metrics import (
    compute_ranking_metrics,
    compute_rating_metrics,
    compute_beyond_accuracy_metrics
)

## Mock Data

We create three DataFrames to demonstrate the metrics:

- **train_data**: Historical user-item interactions (used for beyond-accuracy metrics)
- **ground_truth**: Held-out test set with actual ratings
- **predictions**: Model's predicted scores for user-item pairs

In [5]:
train_data = pandas.DataFrame({
    "user_id": [1, 1, 2, 2, 3, 3],
    "item_id": [101, 102, 103, 104, 105, 106],
    "rating": [5, 4, 3, 5, 4, 3]
})

ground_truth = pandas.DataFrame({
    "user_id": [1, 1, 2, 2, 3, 3],
    "item_id": [103, 104, 105, 106, 107, 108],
    "rating": [5, 4, 5, 3, 4, 5]
})

predictions = pandas.DataFrame({
    "user_id": [1, 1, 2, 2, 3, 3],
    "item_id": [103, 105, 105, 107, 107, 108],
    "prediction": [4.5, 4.0, 4.8, 3.9, 4.1, 3.8]
})

## Ranking Metrics

Evaluate how well the model ranks relevant items at the top of the recommendation list.

| Metric | Description |
|--------|-------------|
| **nDCG@k** | Normalized Discounted Cumulative Gain - measures ranking quality with position-weighted relevance |
| **MAP@k** | Mean Average Precision - average precision across all relevant items |
| **Precision@k** | Fraction of recommended items that are relevant |
| **Recall@k** | Fraction of relevant items that are recommended |

In [6]:
ranking_metrics = compute_ranking_metrics(
    ground_truth=ground_truth,
    predictions=predictions,
    top_k=3
)

print("Ranking metrics:")
for name, value in ranking_metrics.items():
    print(f"  {name}: {value:.4f}")

Ranking metrics:
  ndcg: 0.7421
  map: 0.6667
  precision: 0.4444
  recall: 0.6667


## Rating Metrics

Evaluate how accurately the model predicts actual ratings.

| Metric | Description |
|--------|-------------|
| **RMSE** | Root Mean Squared Error - penalizes large errors more heavily |
| **MAE** | Mean Absolute Error - average absolute difference between predicted and actual |
| **R²** | Coefficient of determination - proportion of variance explained |
| **Explained Variance** | Similar to R², but insensitive to constant prediction offsets |

In [7]:
rating_metrics = compute_rating_metrics(
    ground_truth=ground_truth,
    predictions=predictions
)

print("Rating metrics:")
for name, value in rating_metrics.items():
    print(f"  {name}: {value:.4f}")

Rating metrics:
  rmse: 0.6595
  mae: 0.5000
  r_squared: -1.3200


## Beyond-Accuracy Metrics

Evaluate recommendation quality beyond prediction accuracy - useful for bias analysis.

| Metric | Description |
|--------|-------------|
| **Coverage** | Fraction of catalog items that appear in recommendations (measures popularity bias) |
| **Novelty** | How "obscure" recommended items are based on popularity (higher = more long-tail items) |

In [8]:
beyond_accuracy_metrics = compute_beyond_accuracy_metrics(
    train_dataframe=train_data,
    predictions=predictions
)

print("Beyond-accuracy metrics:")
for name, value in beyond_accuracy_metrics.items():
    print(f"  {name}: {value:.4f}")

Beyond-accuracy metrics:
  coverage: 0.6667
  novelty: 1.2925
  diversity: 1.0000
